# Gemma FT

In [ ]:
import os
import pathlib
from random import randint

import matplotlib.pyplot as plt
import pandas as pd
import pyrootutils
import torch
from datasets import Dataset
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from trl import SFTConfig, SFTTrainer

Path = pathlib.Path

In [ ]:
PROJECT_ROOT = pyrootutils.find_root(
    search_from=os.path.abspath(""), indicator=".project-root"
)

load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

In [ ]:
base_model: str = "google/gemma-3-270m-it"
data_path: Path = PROJECT_ROOT / "notebooks" / "data" / "synthetic" / "ft_data.feather"
checkpoint_dir: Path = PROJECT_ROOT / "notebooks" / "checkpoints"
learning_rate: float = 5e-5

In [ ]:
def create_conversation(sample):
    return {
        "messages": [
            {"role": "user", "content": sample["prompt"]},
            {"role": "assistant", "content": sample["model_response"]},
        ]
    }


# npc_type = "martian"  # @param ["martian", "venusian"]
# dataset = load_dataset("bebechien/MobileGameNPC", npc_type, split="train")

data_df = pd.read_feather(data_path).sample(frac=0.02, random_state=0)
dataset = Dataset.from_pandas(data_df)
dataset = dataset.map(
    create_conversation, remove_columns=dataset.features, batched=False
)
dataset = dataset.train_test_split(test_size=0.2, shuffle=False)
print(dataset["train"][0]["messages"])

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model, dtype="auto", device_map="auto", attn_implementation="eager"
)
tokenizer = AutoTokenizer.from_pretrained(base_model)

print(f"Device: {model.device}")
print(f"DType: {model.dtype}")

In [ ]:
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
rand_idx = randint(0, len(dataset["test"]) - 1)
test_sample = dataset["test"][rand_idx]

prompt = pipe.tokenizer.apply_chat_template(
    test_sample["messages"][:1], tokenize=False, add_generation_prompt=True
)
outputs = pipe(prompt, disable_compile=True)

print(f"Question:\n{test_sample['messages'][0]['content']}\n")
print(f"Original Answer:\n{test_sample['messages'][1]['content']}\n")
print(
    f"Generated Answer (base model):\n{outputs[0]['generated_text'][len(prompt):].strip()}"
)

In [ ]:
torch_dtype = model.dtype

args = SFTConfig(
    output_dir=str(checkpoint_dir),  # directory to save and repository id
    max_length=512,  # max sequence length for model and packing of the dataset
    packing=False,  # Groups multiple samples in the dataset into a single sequence
    num_train_epochs=5,  # number of training epochs
    per_device_train_batch_size=4,  # batch size per device during training
    gradient_checkpointing=False,  # Caching is incompatible with gradient checkpointing
    optim="adamw_torch_fused",  # use fused adamw optimizer
    logging_steps=1,  # log every step
    save_strategy="epoch",  # save checkpoint every epoch
    eval_strategy="epoch",  # evaluate checkpoint every epoch
    learning_rate=learning_rate,  # learning rate
    fp16=True if torch_dtype == torch.float16 else False,  # use float16 precision
    bf16=True if torch_dtype == torch.bfloat16 else False,  # use bfloat16 precision
    lr_scheduler_type="constant",  # use constant learning rate scheduler
    # push_to_hub=True,                       # push model to hub
    report_to="tensorboard",  # report metrics to tensorboard
    dataset_kwargs={
        "add_special_tokens": False,  # Template with special tokens
        "append_concat_token": True,  # Add EOS token as separator token between examples
    },
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
)

In [ ]:
trainer.train()
trainer.save_model()

In [ ]:
# Access the log history
log_history = trainer.state.log_history

# Extract training / validation loss
train_losses = [log["loss"] for log in log_history if "loss" in log]
epoch_train = [log["epoch"] for log in log_history if "loss" in log]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
epoch_eval = [log["epoch"] for log in log_history if "eval_loss" in log]

# Plot the training loss
plt.plot(epoch_train, train_losses, label="Training Loss")
plt.plot(epoch_eval, eval_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model_id = checkpoint_dir

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id, dtype="auto", device_map="auto", attn_implementation="eager"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)


def test(test_sample):
    # Convert as test example into a prompt with the Gemma template
    prompt = pipe.tokenizer.apply_chat_template(
        test_sample["messages"][:1], tokenize=False, add_generation_prompt=True
    )
    outputs = pipe(prompt, max_new_tokens=256, disable_compile=True)

    # Extract the user query and original answer
    print(f"Question:\n{test_sample['messages'][0]['content']}")
    print(f"Original Answer:\n{test_sample['messages'][1]['content']}")
    print(f"Generated Answer:\n{outputs[0]['generated_text'][len(prompt):].strip()}")
    print("-" * 80)


# Test with an unseen dataset
for item in dataset["test"]:
    test(item)

In [ ]:
outputs = pipe(
    [{"role": "user", "content": "Sorry, you are a game NPC."}],
    max_new_tokens=256,
    disable_compile=True,
)
print(outputs[0]["generated_text"][1]["content"])